# xtim — a hands-on tutorial

`xtim` is a [Stim](https://github.com/quantumlib/Stim)-shaped simulator for **logical
magic-state-preparation protocols**. It speaks a strict superset of the Stim circuit
language (the full Clifford set **plus** the non-Clifford `T`/`CS`/`CCZ`, a
`PAULI_EXPECTATION` channel, and classically-controlled Pauli feedback), and it computes
**exactly** the one thing Stim cannot: the magic-state expectation value your protocol
prepares.

This notebook walks the whole arc end to end:

1. Hello, magic state — the one thing Stim can't do
2. `diagnose()` — ask the tool what you built
3. Scoring: frame-correct → post-select → fidelity (and the silent-wrong-answer footgun)
4. Decoder-in-the-loop (DEM + reject region)
5. Sweeping noise with `xtim.collect`
6. Multi-magic — the [[8,3,2]] transversal CCZ

Every code cell below runs as-is. Two cells also use `matplotlib` and `pymatching` —
install everything the notebook needs with `pip install "xtim[tutorial]"` (both cells
degrade gracefully if you skip it). The deeper references are
[`docs/xtim_tour.md`](../docs/xtim_tour.md), [`docs/xtim_dialect.md`](../docs/xtim_dialect.md),
and [`docs/xtim_dem_reject.md`](../docs/xtim_dem_reject.md).

In [ ]:
import numpy as np
import xtim

print("xtim", xtim.__version__)
# The bundled demo circuits load straight from the installed package — no file
# paths, no repo clone. (For your own circuit: xtim.Circuit.from_file("you.stim").)
print("bundled examples:", xtim.list_examples())

## 1. Hello, magic state — `T|+⟩`

A single `T` on a `|+⟩` state. Stim can't even parse `T` (it's non-Clifford); xtim computes
the resulting Pauli expectations exactly. `T|+⟩` is the canonical magic state, with
`⟨X⟩ = ⟨Y⟩ = 1/√2 ≈ 0.7071`.

Two things to notice: we **declare** the observables we care about with `PAULI_EXPECTATION`,
and each shot returns the *exact* expectation value (a real number), not a ±1 sample.

In [ ]:
hello = xtim.Circuit('''
RX 0          # |+>
T 0           # -> T|+>, a magic state
PAULI_EXPECTATION(0) X0
PAULI_EXPECTATION(1) Y0
''')

dets, exps = hello.compile_detector_sampler(seed=0).sample(10_000, return_expectations=True)
print("measured <X>, <Y> =", np.round(exps.mean(axis=0), 4))
print("exact   <X>, <Y> =", np.round([2**-0.5, 2**-0.5], 4))
print("per-shot value is exact (std ~ 0):", np.round(exps.std(axis=0), 6))

## 2. `diagnose()` — ask the tool what you built

Point `diagnose()` at a protocol and it does one noiseless run and **reports the facts you
would otherwise reverse-engineer by hand** — never interpreting them for you. We use a
bundled demo: a teleported `T|+⟩` into a small repetition code.

Read off three things:

- **deterministic detectors** — always 0 noiselessly; these are what you post-select on.
- **signed beta** — the target `⟨Pᵢ⟩` your magic state prepares (magnitude + sign).
- **byproduct frame** — when a column's sign flips shot-to-shot, `diagnose()` hands you the
  record-parity `HINT` that predicts it. That hint is the per-shot sign rule.

The first tiny `sample()` below just warms the reference cache so the report shows `χ` as
`cached`. It is optional — without it, `diagnose()` still works and prints
`reference chi : not yet cached` (same facts, just not memoized yet).

In [ ]:
c = xtim.load_example("miniature_oracle")
c.compile_detector_sampler(seed=0).sample(64, return_expectations=True)  # warm the cache (optional)
report = c.diagnose(shots=2000, seed=0)
print(report)

## 3. Scoring: frame-correct → post-select → fidelity

`diagnose()` reports; **you** score. The recipe is plain NumPy, but there is one
silent-wrong-answer trap, so we do it carefully.

Each `byproduct frame` column's sign flips ~50% of the time. If you average the raw
expectations you get ≈ 0 → a fidelity of ≈ 0.5. You must first **flip each shot's sign by
the parity of the `frame_hint` records** (`diagnose()` discovered them). *Only* the parity —
the trailing `+1` constant in the report is the canonical sign, already folded into the
reported signed beta; do **not** re-apply it per shot.

In [ ]:
N = 20_000
ncols = len(report.expectations)
keep_dets = report.deterministic_detectors
frame_records = [e.frame_hint for e in report.expectations]

dets, exps, meas = c.compile_detector_sampler(seed=7).sample(
    N, return_expectations=True, return_measurements=True)

# (a) post-select the deterministic detectors
keep = ~dets[:, keep_dets].any(axis=1)

# (b) frame-correct: flip each shot by the parity of its frame_hint records
flips = np.zeros((N, ncols), dtype=bool)
for j, records in enumerate(frame_records):
    if records:
        flips[:, j] = (meas[:, records].sum(axis=1) % 2).astype(bool)
corrected = np.where(flips, -exps, exps)

a_bar      = corrected[keep].mean(axis=0)    # frame-corrected, post-selected
a_bar_raw  = exps[keep].mean(axis=0)         # the footgun: no frame correction

# (c) combine with YOUR target (T|+> in a rep code -> +|beta| on both columns)
beta = np.array([e.abs_value for e in report.expectations])   # |beta|, signs +1
F      = (1 + beta @ a_bar) / 2
F_raw  = (1 + beta @ a_bar_raw) / 2

print(f"kept {keep.sum()}/{N}")
print(f"a-bar (frame-corrected) = {np.round(a_bar, 4)}")
print(f"F = {F:.4f}     (without the frame hint: {F_raw:.4f}  <- the ~0.5 footgun)")
assert abs(np.linalg.norm(beta)) <= 1 + 1e-9, "target must be physical: |beta| <= 1"


Two guardrails worth internalizing:

- **The frame hint does real work.** Dropping it collapses `F` to ≈ 0.5 (the byproduct
  randomizes the sign). That's the single most common mistake.
- **`F > 1` means a bad target, not noise.** Because the per-shot expectations are *exact*,
  with a physical target (`|beta| ≤ 1`) the fidelity is an average of per-shot overlaps and
  can't exceed 1 except by floating point. So `F > 1` is a reliable signal your coefficients
  are unphysical — fix the target, not the statistics.

## 4. Decoder-in-the-loop (DEM + reject region)

For a real protocol you decode the Pauli-correctable faults and post-select the rest.
`detector_error_model_with_reject()` hands you a clean Stim DEM (`res.dem`) plus the
not-Pauli-correctable faults to flag. We use `cultivation_d3_faithful` and feed the DEM to
`pymatching`.

In [ ]:
cult = xtim.load_example("cultivation_d3_faithful")
res = cult.detector_error_model_with_reject()
print("decodable DEM mechanisms:", sum(l.startswith("error(") for l in str(res.dem).splitlines()))
print("flagged not-correctable :", len(res.postselect_faults))
print("conservative reject set :", len(res.reject_detectors), "detectors")

dets, _obs, exps = cult.compile_detector_sampler(seed=1).sample(
    20_000, separate_observables=True, return_expectations=True)
keep = res.keep_mask(dets)                       # blunt conservative post-select
print(f"\nacceptance: {keep.mean():.3f}")
print("magic value, post-selected:", np.round(exps[keep].mean(axis=0), 4), " (target +0.7071)")

try:
    import pymatching
    m = pymatching.Matching.from_detector_error_model(res.dem)
    corr = m.decode_batch(dets[keep])
    print(f"decoded {corr.shape[0]} accepted shots "
          f"({m.num_fault_ids} obs, {m.num_detectors} detectors)")
except ImportError:
    print("(install pymatching to decode res.dem)")

## 5. Sweeping noise with `xtim.collect`

`xtim.collect` runs a list of `Task`s in parallel and returns one stats row each. Here we
sweep the physical error rate `p` and plot the acceptance rate and the post-selected magic
value. Each `Task(p=...)` rescales the circuit's baked-in noise to `p`.

In [ ]:
keep_fn = lambda dets, meas: res.keep_mask(dets)   # reuse cultivation's reject set
ps = [3e-4, 1e-3, 3e-3, 1e-2]
tasks = [xtim.Task(cult, p=p, shots=20_000, seed=2, keep=keep_fn, metadata={"p": p})
         for p in ps]
rows = xtim.collect(tasks, num_workers=2)

acc = [r["acceptance_rate"] for r in rows]
val = [float(np.mean(r["value"])) for r in rows]      # mean over the two magic columns
for r in rows:
    print(f"p={r['metadata']['p']:.0e}  accept={r['acceptance_rate']:.3f}  "
          f"value={np.round(r['value'],4)}")

# The plot is optional eye-candy; the numbers above are the substance.
try:
    import matplotlib.pyplot as plt
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
    a1.plot(ps, acc, "o-"); a1.set_xscale("log"); a1.set_xlabel("p"); a1.set_ylabel("acceptance")
    a2.plot(ps, val, "s-"); a2.set_xscale("log"); a2.set_xlabel("p")
    a2.set_ylabel("post-selected magic value"); a2.axhline(2**-0.5, ls="--", c="grey")
    fig.tight_layout(); plt.show()
except ImportError:
    print("\n(install matplotlib — or `pip install \"xtim[tutorial]\"` — to see the plot)")

### 5b. The fidelity directly — `target_k`

We just plotted the raw post-selected *value*. To get an actual **fidelity** with an
error bar, add one ingredient: `target_k` — the number of logical qubits in the target
magic state (here 1). `collect` then derives the target and does the Bloch arithmetic,
returning `fidelity`, `infidelity` (= 1−F), and a correlation-aware `infidelity_sem` per row.

In [ ]:
ftasks = [xtim.Task(cult, p=p, shots=20_000, seed=2, keep=keep_fn, target_k=1)
          for p in ps]
for r in xtim.collect(ftasks, num_workers=2):
    print(f"p={r['p']:.0e}  accept={r['acceptance_rate']:.3f}  "
          f"F={r['fidelity']:.4f}  1-F={max(0.0, r['infidelity']):.2e}")

Here the conservative post-selection heralds *every* detectable error, so the **accepted**
state is essentially perfect (`F ≈ 1`) and **acceptance** is the price — it falls with `p`.
To watch `1−F` actually *rise* with noise (the usual distillation curve), either skip
post-selection or decode the residual logical errors. The standalone
[`fidelity_curve.py`](fidelity_curve.py) does exactly that on a noisy `|T⟩` and plots
`1−F ± sem` vs `p`.

## 6. Multi-magic — the [[8,3,2]] transversal CCZ

Everything so far prepared a single logical magic state (χ ≤ 2). xtim is exact at any χ.
The [[8,3,2]] cube color code has a **transversal CCZ** on **3** logical qubits — its output
is `CCZ|+++⟩_L`, a genuine multi-magic state; its compiled reference has rank χ = 8 (xtim conditions on
the code's syndrome, so this exceeds the bare CCZ state's rank). Each logical-X marginal
of that state is `+0.5`.

The prep measures the code's syndrome and **feedback-corrects** it before the transversal
`T`/`T_DAG` (xtim coherentizes feedback that crosses the non-Clifford gate). The result is a
deterministic prep: 100% acceptance, no post-selection — and the χ=8 reference compiles and
caches like any other.

In [ ]:
cube = xtim.load_example("cube_ccz")
exps = cube.compile_detector_sampler(seed=5).sample(20_000, return_expectations=True)[1]
print("<Xbar_1>, <Xbar_2>, <Xbar_3> =", np.round(exps.mean(axis=0), 4), " (CCZ|+++> marginals)")
print("deterministic (std ~ 0):", np.round(exps.std(axis=0), 6))
print("reference chi:", cube.diagnose(shots=200).chi)   # 8 — compiled & cached

## Where to go next

- **The dialect** — every added instruction (`T`/`CS`/`CCZ`, `PAULI_EXPECTATION`, feedback,
  `MPP`/`SPP`): [`docs/xtim_dialect.md`](../docs/xtim_dialect.md).
- **The lean decoder workflow**, executed verbatim by the test suite:
  [`docs/xtim_tour.md`](../docs/xtim_tour.md).
- **Decoder DEMs with a reject region**: [`docs/xtim_dem_reject.md`](../docs/xtim_dem_reject.md).
- **Score your own protocol** (no bundled lore): `examples/onboarding_new_protocol.py`.

**Scope & limits.** xtim is a magic-state-prep co-processor, not a Stim replacement: its
sweet spot is protocols whose output is a single low-rank logical magic state. For
pure-Clifford memory sweeps, keep using Stim — pair xtim (magic) with Stim (bulk Clifford).